## Imports And Model Loading

In [ ]:
import os
from typing import Optional
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_groq import ChatGroq
import json

# Set Groq API key
os.environ["GROQ_API_KEY"] = "KEY"

## Info Class

In [ ]:
class PersonInfo(BaseModel):
    name: Optional[str] = Field(default=None, description="Name of the person. Return null if not found.")
    email: Optional[str] = Field(default=None, description="Email address. Return null if not found.")
    Age: Optional[str] = Field(default=None, description="Age of the person. Return null if not found.")
    Faculty: Optional[str] = Field(default=None, description="Faculty or university department. Return null if not found.")

## Model + Prpmot 

In [ ]:
def extract_information(text: str):
    # Using Groq model - llama-3.3-70b-versatile
    llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

    parser = JsonOutputParser(pydantic_object=PersonInfo)

    prompt = PromptTemplate(
        template="Extract the specified information from the text.\n{format_instructions}\n\nText: {text}\n",
        input_variables=["text"],
        partial_variables={"format_instructions": parser.get_format_instructions()},
    )

    chain = prompt | llm | parser

    try:
        result = chain.invoke({"text": text})
        
        print(f"Input Text: {text}")
        print("-" * 30)
        print("Valid JSON Output Extracted:")
        print(json.dumps(result, indent=2))
        print("\n" + "=" * 30 + "\n")
        return result
    
    except Exception as e:
        print(f"Failed to parse or validate: {e}")
        return None

## Test Case

In [ ]:
# Ask the user to enter text
user_text = input("Please enter the text you want to extract information from: ")

# Check if input is not empty and then run extraction
if user_text.strip():
    extract_information(user_text)
    
else:
    print("No text entered. Please try again.")

Input Text: Hello my name is Nour ,i'm 22.
------------------------------
Valid JSON Output Extracted:
{
  "name": "Nour",
  "email": null,
  "Age": "22",
  "Faculty": null
}


